In [5]:
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import TensorDataset, DataLoader, WeightedRandomSampler
import numpy as np
from sklearn.metrics import f1_score, precision_score, recall_score
from sklearn.preprocessing import StandardScaler
import pandas as pd
from sklearn.model_selection import train_test_split

def parse_vector(vec_str):
    # Splitta la stringa su ";" e converte ciascun valore in float
    floats = [float(num_str) for num_str in vec_str.split(";")]
    return floats


df= pd.read_csv("all_audio_wav2vec_dataset_complete.csv")
#print(df["class"].unique())
df['class'] = df['class'].apply(lambda x: 1 if ((x == 'ambulance') or (x == 'firetruck') or (x == 'police') or (x == 'siren') ) else 0) #sostituisco 1 a tutti i tipi di sirene, 0 al resto

df=df[df["snr"].isin([20])]
X_series = df["vector"].apply(parse_vector)
X = np.array(X_series.tolist(), dtype=np.float32)
y = df["class"].values.astype(np.float32)

X_train, X_test, y_train, y_test = train_test_split(X,y, test_size = 0.30, stratify=y) #stratify mantiene la proporzione rispetto IMPORTANTE
X_train, X_validation, y_train, y_validation = train_test_split(X_train,y_train,test_size=0.30, stratify = y_train)

# -------------------------------------------------------
# 1. Classe DynamicNet con Dropout
# -------------------------------------------------------
class DynamicNet(nn.Module):
    def __init__(self, layer_sizes, dropout_prob=0.0):
        """
        layer_sizes: lista di interi che rappresentano il numero di neuroni
                     in ciascun layer. [768, 256, 128, 1] per esempio.
        dropout_prob: probabilità di dropout da applicare sui layer intermedi.
        """
        super(DynamicNet, self).__init__()
        self.layers = nn.ModuleList()
        self.dropouts = nn.ModuleList()  # Per gestire dropout tra i layer
        for i in range(len(layer_sizes) - 1):
            in_features = layer_sizes[i]
            out_features = layer_sizes[i + 1]
            self.layers.append(nn.Linear(in_features, out_features))

            # Aggiungi un livello di Dropout (tranne dopo l'ultimo layer)
            if i < len(layer_sizes) - 2:
                self.dropouts.append(nn.Dropout(p=dropout_prob))
            else:
                self.dropouts.append(nn.Identity())  # Nessun dropout sull'ultimo livello

    def forward(self, x):
        for i in range(len(self.layers) - 1):
            x = self.layers[i](x)
            x = F.relu(x)
            x = self.dropouts[i](x)
        # Ultimo layer con Sigmoid per classificazione binaria
        x = torch.sigmoid(self.layers[-1](x))
        return x

# -------------------------------------------------------
# 2. Early Stopping con min_delta
# -------------------------------------------------------
class EarlyStopping:
    """
    Ferma l'allenamento se la metrica monitorata (es. validation loss) 
    non migliora di almeno `min_delta` per `patience` epoche consecutive.
    """
    def __init__(self, patience=5, min_delta=0.0):
        self.patience = patience
        self.min_delta = min_delta
        self.best_score = None
        self.counter = 0
        self.early_stop = False

    def __call__(self, current_score):
        # Se best_score è None, inizializzo
        if self.best_score is None:
            self.best_score = current_score
            return
        
        # Se il miglioramento è minore di min_delta
        if (self.best_score - current_score) < self.min_delta:
            self.counter += 1
            if self.counter >= self.patience:
                self.early_stop = True
        else:
            self.best_score = current_score
            self.counter = 0


# -------------------------------------------------------
# 3. Funzione di training adattata per EarlyStopping
# -------------------------------------------------------
def train_model(model, 
                train_loader, 
                val_loader, 
                epochs=10, 
                lr=1e-3, 
                weight_decay=1e-4,
                early_stop_patience=5,
                early_stop_min_delta=0.0):
    """
    Esegue il loop di training e validazione su un singolo modello,
    con Early Stopping, e restituisce il modello addestrato.
    """
    criterion = nn.BCELoss()  # Per un problema binario
    # Usando SGD con L2 (weight_decay) per regolarizzazione
    optimizer = optim.SGD(model.parameters(), lr=lr, weight_decay=weight_decay)

    early_stopper = EarlyStopping(patience=early_stop_patience, min_delta=early_stop_min_delta)

    for epoch in range(epochs):
        model.train()
        running_loss = 0.0
        
        for X_batch, y_batch in train_loader:
            X_batch = X_batch.to(device)
            y_batch = y_batch.to(device)

            # Forward pass
            outputs = model(X_batch)
            loss = criterion(outputs, y_batch.unsqueeze(1))
            
            # Backward pass e ottimizzazione
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            
            running_loss += loss.item()

        # Calcolo della loss media su epoca
        train_loss = running_loss / len(train_loader)

        # -------------------------------------------
        # Valutazione su validation set
        # -------------------------------------------
        model.eval()
        val_loss = 0.0
        with torch.no_grad():
            for X_val, y_val in val_loader:
                X_val = X_val.to(device)
                y_val = y_val.to(device)

                val_outputs = model(X_val)
                loss_val = criterion(val_outputs, y_val.unsqueeze(1))
                val_loss += loss_val.item()

        val_loss = val_loss / len(val_loader)

        print(f"Epoch [{epoch+1}/{epochs}] - "
              f"Train Loss: {train_loss:.4f}  |  Validation Loss: {val_loss:.4f}")

        # Controllo EarlyStopping
        early_stopper(val_loss)
        if early_stopper.early_stop:
            print(f"** Early Stopping attivato all'epoch {epoch+1} **")
            break

    return model


# -------------------------------------------------------
# 4. Esempio di integrazione di TUTTE le fasi
# -------------------------------------------------------
# Esempio di configurazioni di layer:
module_layer_sizes = [
        [768, 512, 1], # 1 hidden layer con 512 neuroni
        [768, 256, 1], # 1 hidden layer con 256 neuroni
        [768, 128, 1], # 1 hidden layer con 128 neuroni
        [768, 64, 1],  # 1 hidden layer con 64 neuroni
        [768, 32, 1],  # 1 hidden layer con 32 neuroni
        [768, 512, 256, 1],   # 2 hidden layer (512 e 256 neuroni)
        [768, 256, 128, 1],   # 2 hidden layer (256 e 128 neuroni)
        [768, 128, 64, 1], # 2 hidden layer (128 e 64 neuroni)
        [768, 64, 32, 1],    # 2 hidden layer (64 e 32 neuroni)
        [768, 256, 128, 64, 1],  # 3 hidden layer (256, 128 e 64 neuroni)
        [768, 128, 64, 32, 1],  # 3 hidden layer (128, 64 e 32 neuroni)
        [768, 512, 256, 128, 64, 1], # 4 hidden layer (512, 256, 128 e 64 neuroni)
        [768, 512, 256, 128, 64, 32, 1], # 5 hidden layer (512, 256, 128, 64, 32 neuroni)
        [768, 512, 256, 128, 64, 32, 16, 1] # 6 hidden layer (512, 256, 128, 64, 32, 16 neuroni)
    ]

# 4.1 - Normalizzazione (StandardScaler) su X_train, X_val, X_test
scaler = StandardScaler()
X_train_norm = scaler.fit_transform(X_train)
X_val_norm   = scaler.transform(X_validation)
X_test_norm  = scaler.transform(X_test)

# 4.2 - Conversione in tensori PyTorch
device = "cuda" if torch.cuda.is_available() else "cpu"

X_train_t = torch.tensor(X_train_norm, dtype=torch.float32)
y_train_t = torch.tensor(y_train,      dtype=torch.float32)
X_val_t   = torch.tensor(X_val_norm,   dtype=torch.float32)
y_val_t   = torch.tensor(y_validation, dtype=torch.float32)
X_test_t  = torch.tensor(X_test_norm,  dtype=torch.float32)
y_test_t  = torch.tensor(y_test,       dtype=torch.float32)

# 4.3 - Creazione Dataset
train_dataset = TensorDataset(X_train_t, y_train_t)
val_dataset   = TensorDataset(X_val_t,   y_val_t)
test_dataset  = TensorDataset(X_test_t,  y_test_t)

# 4.4 - WeightedRandomSampler per il train set (se sbilanciato)
class_counts = np.bincount(y_train.astype(int))
total_samples = len(y_train)
weight_per_class = [total_samples / c if c > 0 else 0 for c in class_counts]

weights = [weight_per_class[int(label)] for label in y_train]
weights = torch.tensor(weights, dtype=torch.float32)

sampler = WeightedRandomSampler(
    weights=weights,
    num_samples=len(weights),
    replacement=True  # tipicamente True
)

# 4.5 - DataLoader con batch_size e WeightedRandomSampler
batch_size = 32
train_loader = DataLoader(train_dataset, batch_size=batch_size, sampler=sampler)
val_loader   = DataLoader(val_dataset,   batch_size=batch_size, shuffle=False)
test_loader  = DataLoader(test_dataset,  batch_size=batch_size, shuffle=False)

# 4.6 - Ricerca del best model su diverse configurazioni di layer
best_model = None
best_val_loss = float("inf")
criterion = nn.BCELoss()  # verrà usata per la stima su validation

for i, layer_config in enumerate(module_layer_sizes):
    print(f"\n>>> Training model configuration #{i+1}: {layer_config}")

    # Inizializza un modello con dropout
    model = DynamicNet(layer_config, dropout_prob=0.25).to(device)

    # Addestra il modello con train_model
    model = train_model(
        model, 
        train_loader, 
        val_loader, 
        epochs=150,        
        lr=0.001,          # learning rate
        weight_decay=1e-3, # L2 reg
        early_stop_patience=15,
        early_stop_min_delta=1e-4
    )
    
    # Valuta la loss di validazione finale
    model.eval()
    running_val_loss = 0.0
    with torch.no_grad():
        for X_val, y_val in val_loader:
            X_val = X_val.to(device)
            y_val = y_val.to(device)
            
            outputs = model(X_val)
            loss_val = criterion(outputs, y_val.unsqueeze(1))
            running_val_loss += loss_val.item()

    avg_val_loss = running_val_loss / len(val_loader)
    print(f"Validation Loss (final) per config #{i+1}: {avg_val_loss:.4f}")

    # Se la loss corrente è migliore, salviamo il modello come best_model
    if avg_val_loss < best_val_loss:
        best_val_loss = avg_val_loss
        best_model = model

# 4.7 - Valutazione finale del best_model sul test set
best_model.eval()
print("Configurazione best model")
print(best_model)
all_preds = []
all_labels = []
with torch.no_grad():
    for X_test_batch, y_test_batch in test_loader:
        X_test_batch = X_test_batch.to(device)
        y_test_batch = y_test_batch.to(device)

        preds = best_model(X_test_batch)
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(y_test_batch.cpu().numpy())

all_preds = np.array(all_preds).flatten()
all_labels = np.array(all_labels).flatten()

# Soglia a 0.5 per classificazione binaria
binary_preds = (all_preds >= 0.5).astype(int)

f1 = f1_score(all_labels, binary_preds)
precision = precision_score(all_labels, binary_preds)
recall = recall_score(all_labels, binary_preds)

print(f"\nMiglior modello trovato con Validation Loss = {best_val_loss:.4f}")
print(f"F1 su Test set: {f1:.4f}")
print(f"Precision su Test set: {precision:.4f}")
print(f"Recall su Test set: {recall:.4f}")



>>> Training model configuration #1: [768, 512, 1]
Epoch [1/150] - Train Loss: 0.6585  |  Validation Loss: 0.6307
Epoch [2/150] - Train Loss: 0.6070  |  Validation Loss: 0.5919
Epoch [3/150] - Train Loss: 0.5653  |  Validation Loss: 0.5647
Epoch [4/150] - Train Loss: 0.5387  |  Validation Loss: 0.5298
Epoch [5/150] - Train Loss: 0.5053  |  Validation Loss: 0.5113
Epoch [6/150] - Train Loss: 0.4947  |  Validation Loss: 0.4903
Epoch [7/150] - Train Loss: 0.4766  |  Validation Loss: 0.4673
Epoch [8/150] - Train Loss: 0.4570  |  Validation Loss: 0.4567
Epoch [9/150] - Train Loss: 0.4434  |  Validation Loss: 0.4509
Epoch [10/150] - Train Loss: 0.4357  |  Validation Loss: 0.4402
Epoch [11/150] - Train Loss: 0.4261  |  Validation Loss: 0.4204
Epoch [12/150] - Train Loss: 0.4138  |  Validation Loss: 0.4091
Epoch [13/150] - Train Loss: 0.4155  |  Validation Loss: 0.4078
Epoch [14/150] - Train Loss: 0.3999  |  Validation Loss: 0.3995
Epoch [15/150] - Train Loss: 0.3900  |  Validation Loss: 0.39